## Integrando todo: funcion reutilizable

Hasta aca trabajaste cada paso por separado. Ahora vamos a encapsular el flujo completo en una funcion que reciba una URL de YouTube y devuelva texto listo para procesar. Esta clase de integracion importa porque transforma una serie de pruebas manuales en un pipeline reutilizable para construir corpus de manera sistematica.


In [ ]:
def youtube_a_texto(url: str, modelo_whisper: str = "small", idioma: str = "es", output_dir: str = "corpus") -> dict:
    # Pipeline completo: URL de YouTube -> audio -> transcripcion -> texto
    import json
    import os
    import re
    import shutil
    import whisper
    import yt_dlp

    os.makedirs(output_dir, exist_ok=True)

    if "preparar_ffmpeg" in globals():
        _, ffmpeg_dir, _ = preparar_ffmpeg()
    else:
        ruta_ffmpeg = os.environ.get("FFMPEG_PATH") or shutil.which("ffmpeg")
        if not ruta_ffmpeg or not os.path.exists(ruta_ffmpeg):
            raise FileNotFoundError("No se encontro ffmpeg para ejecutar el pipeline completo.")
        ffmpeg_dir = os.path.dirname(ruta_ffmpeg)
        if ffmpeg_dir not in os.environ.get("PATH", ""):
            os.environ["PATH"] = ffmpeg_dir + os.pathsep + os.environ.get("PATH", "")

    ydl_opts = {
        "format": "bestaudio/best",
        "noplaylist": True,
        "ffmpeg_location": ffmpeg_dir,
        "postprocessors": [
            {
                "key": "FFmpegExtractAudio",
                "preferredcodec": "mp3",
                "preferredquality": "192",
            }
        ],
        "outtmpl": f"{output_dir}/%(title)s.%(ext)s",
    }

    if "sanitizar_nombre_archivo" in globals():
        sanitizar = sanitizar_nombre_archivo
    else:
        def sanitizar(ruta_o_nombre: str, max_len: int = 80) -> str:
            nombre = os.path.splitext(os.path.basename(ruta_o_nombre))[0]
            invalidos = set('<>:/\\|?*') | {chr(34)}
            nombre = ''.join('_' if caracter in invalidos or ord(caracter) < 32 else caracter for caracter in nombre)
            nombre = re.sub(r'\s+', ' ', nombre).strip().rstrip('. ')
            return (nombre[:max_len] or 'transcripcion').strip()

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        titulo = info["title"]
        duracion = info["duration"]
        audio_generado = os.path.abspath(ydl.prepare_filename(info).rsplit(".", 1)[0] + ".mp3")

    print(f"Audio descargado: {titulo} ({duracion} segundos)")

    cache_modelos = globals().setdefault("_MODELOS_WHISPER", {})
    if modelo_whisper not in cache_modelos:
        cache_modelos[modelo_whisper] = whisper.load_model(modelo_whisper)
    modelo = cache_modelos[modelo_whisper]
    resultado_local = modelo.transcribe(audio_generado, language=idioma, fp16=False)
    print(f"Transcripcion completa: {len(resultado_local['segments'])} segmentos")

    nombre_base = sanitizar(audio_generado)
    ruta_txt = os.path.join(output_dir, f"{nombre_base}.txt")
    with open(ruta_txt, "w", encoding="utf-8") as archivo_txt:
        archivo_txt.write(resultado_local["text"])

    ruta_json = os.path.join(output_dir, f"{nombre_base}.json")
    with open(ruta_json, "w", encoding="utf-8") as archivo_json:
        json.dump(
            {
                "fuente": url,
                "titulo": titulo,
                "duracion_segundos": duracion,
                "idioma": resultado_local["language"],
                "texto": resultado_local["text"],
                "segmentos": [
                    {"inicio": segmento["start"], "fin": segmento["end"], "texto": segmento["text"].strip()}
                    for segmento in resultado_local["segments"]
                ],
            },
            archivo_json,
            ensure_ascii=False,
            indent=2,
        )

    return {
        "titulo": titulo,
        "duracion": duracion,
        "texto": resultado_local["text"],
        "segmentos": resultado_local["segments"],
        "archivos_generados": [audio_generado, ruta_txt, ruta_json],
    }


# Uso sugerido:
# corpus = youtube_a_texto("https://www.youtube.com/watch?v=yX2tPSjBoeU")
# print(corpus["texto"][:500])


## Descarga por lotes - Construir un corpus desde una lista de URLs

Cuando pasas de un solo archivo a varios videos, cambias de escala: ya no estas resolviendo una descarga puntual sino construyendo un corpus. En ese contexto aparecen problemas nuevos, como control de errores, consistencia de formatos, almacenamiento y trazabilidad. Procesar por lotes te permite convertir una lista de URLs en una coleccion de textos comparable y reutilizable.


In [ ]:
urls = [
    "https://www.youtube.com/watch?v=yX2tPSjBoeU",
    "https://www.youtube.com/watch?v=ruepxLoEwoo",
    "https://www.youtube.com/watch?v=WqXr0AujesY",
]

resultados = []
errores = []

for indice, url in enumerate(urls, start=1):
    print("\n" + "=" * 60)
    print(f"Procesando {indice}/{len(urls)}: {url}")
    print("=" * 60)
    try:
        resultado_lote = youtube_a_texto(url, modelo_whisper="small")
        resultados.append(resultado_lote)
        print(f"OK: {resultado_lote['titulo']} - {len(resultado_lote['texto'])} caracteres")
    except Exception as error:
        errores.append({"url": url, "error": str(error)})
        print(f"ERROR: {error}")

print(f"\nResumen: {len(resultados)} exitosos, {len(errores)} errores")

corpus_total = "\n\n---\n\n".join(resultado_lote["texto"] for resultado_lote in resultados)
print(f"Corpus total: {len(corpus_total)} caracteres, ~{len(corpus_total.split())} palabras")
